# Vehicle Type Classification — Naive Bayes

Runs end to end and writes `submissions/submission_naive_bayes.csv`.

Gaussian Naive Bayes assumes features are independent given the class. Here 14 feature pairs have |r| > 0.9, so the assumption is badly violated and plain Naive Bayes is the weakest model. We also test two preprocessing variants (power transform; PCA-whitening, which decorrelates the features, so the independence assumption holds better) and tune var_smoothing.

## Setup and preprocessing
* Drop `vehicle_id` (identifier only).
* Drop the 2,500 exact duplicate training rows: they would leak across CV folds and inflate scores (labels never conflict, so nothing is lost).
* No imputation (no missing values), no outlier removal (bounded values), no resampling (balanced classes).
* Tuned choice of preprocessing: none / Yeo-Johnson power transform / standardise + PCA-whitening (decorrelation).

In [1]:
import os, json, numpy as np, pandas as pd, warnings
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
os.makedirs("submissions", exist_ok=True)
SEED = 42; np.random.seed(SEED)
for d in ["data", ".", "..", "/kaggle/input/vehicle-type-classification"]:
    if os.path.exists(f"{d}/vehicle_train_10000.csv"): DATA_DIR = d; break
else: raise FileNotFoundError("Put the competition CSVs next to this notebook or in ./data")
train  = pd.read_csv(f"{DATA_DIR}/vehicle_train_10000.csv")
test   = pd.read_csv(f"{DATA_DIR}/vehicle_test.csv")
sample = pd.read_csv(f"{DATA_DIR}/vehicle_sample_submission.csv")
TARGET, ID = "vehicle_class", "vehicle_id"
FEATURES = [c for c in test.columns if c != ID]
df = train.drop_duplicates(subset=FEATURES).reset_index(drop=True)   # remove 2,500 exact duplicate rows
X, y, X_test = df[FEATURES], df[TARGET].values, test[FEATURES]
CV5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
print("training rows after de-dup:", len(df), "| class counts:", np.bincount(y))

training rows after de-dup: 7500 | class counts: [2539 2490 2471]


## Hyper-parameter tuning (5-fold stratified CV, accuracy)

In [2]:
from sklearn.pipeline import Pipeline
pipe = Pipeline([("prep", "passthrough"), ("nb", GaussianNB())])
space = {"prep": ["passthrough", PowerTransformer(), make_pipeline(StandardScaler(), PCA(whiten=True, random_state=SEED))],
         "nb__var_smoothing": np.logspace(-11, 1, 13)}
search = GridSearchCV(pipe, space, cv=CV5, scoring="accuracy", n_jobs=2).fit(X, y)
r = pd.DataFrame(search.cv_results_)
r["prep_name"] = r["param_prep"].astype(str).str[:28]
print(r.groupby("prep_name")["mean_test_score"].max().round(4).to_string())

prep_name
Pipeline(steps=[('standardsc    0.8628
PowerTransformer()              0.8492
passthrough                     0.8475


### Plain Gaussian Naive Bayes (no decorrelation) for comparison

In [3]:
plain = GridSearchCV(GaussianNB(), {"var_smoothing": np.logspace(-12, -1, 12)}, cv=CV5, scoring="accuracy").fit(X, y)
print("plain GaussianNB CV acc:", round(plain.best_score_, 4), "-> shows the cost of the independence assumption on correlated features")
pd.DataFrame({ID: test[ID], TARGET: plain.best_estimator_.fit(X, y).predict(X_test).astype(int)}).to_csv("submissions/submission_naive_bayes_plain.csv", index=False)

plain GaussianNB CV acc: 0.8475 -> shows the cost of the independence assumption on correlated features


## Stability check, overfitting check, and submission

In [4]:
best = search.best_estimator_
print("best params:", search.best_params_, "| 5-fold CV acc:", round(search.best_score_, 4))
rep = cross_val_score(best, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=SEED + 1), scoring="accuracy", n_jobs=1)
oof = cross_val_predict(best, X, y, cv=CV5)
print(f"Repeated CV accuracy: {rep.mean():.4f} ± {rep.std():.4f} | OOF macro-F1: {f1_score(y, oof, average='macro'):.4f}")
best.fit(X, y)
print("train accuracy:", round(accuracy_score(y, best.predict(X)), 4), "vs CV", round(search.best_score_, 4))
pred = best.predict(X_test).astype(int)
sub = pd.DataFrame({ID: test[ID], TARGET: pred})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample) and (sub[ID].values == sample[ID].values).all() and set(sub[TARGET]) <= {0, 1, 2}
sub.to_csv("submissions/submission_naive_bayes.csv", index=False)
json.dump(dict(model="Naive Bayes", cv_acc=float(search.best_score_), repeated_cv=float(rep.mean()), repeated_cv_std=float(rep.std()),
               macro_f1=float(f1_score(y, oof, average="macro")), train_acc=float(accuracy_score(y, best.predict(X))), params=str(search.best_params_)), open("submissions/submission_naive_bayes_result.json", "w"))
print("written submission_naive_bayes.csv | predicted class counts:", np.bincount(pred, minlength=3).tolist())

best params: {'nb__var_smoothing': np.float64(1.0), 'prep': Pipeline(steps=[('standardscaler', StandardScaler()),
                ('pca', PCA(random_state=42, whiten=True))])} | 5-fold CV acc: 0.8628


Repeated CV accuracy: 0.8634 ± 0.0087 | OOF macro-F1: 0.8625
train accuracy: 0.8656 vs CV 0.8628
written submission_naive_bayes.csv | predicted class counts: [812, 806, 882]
